# 03 · Comparative analysis → paper tables & figures
Reads `results/`, produces every table/figure the paper includes (written to `paper/tables`, `paper/figures`).
Headline: **geometric-mean relative MASE vs seasonal naive** (lower better) with bootstrap 95% CIs over tasks; ranks via Friedman + Nemenyi.

In [ ]:
import os, sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, yaml
pd.set_option("display.width", 160, "display.max_columns", 30)

In [ ]:
from tscompare import aggregate as ag
from tscompare.analysis import plots
import matplotlib.pyplot as plt
RES = "results" if Path("results").exists() and any(Path("results").rglob("*.json")) else "/tmp/smoke_nb"
print("reading", RES)
df = ag.load_results(RES)
df.groupby(["track", "model"]).status.value_counts().unstack(fill_value=0)

## 1 · Main table (all methods, same tasks)

In [ ]:
tbl = ag.summary_table(df, metrics=("mase", "wql"))
show = ["family", "track", "mase_rel", "mase_ci_lo", "mase_ci_hi", "wql_rel", "skill_%", "avg_rank", "win_rate_%", "tasks_ok", "tasks_failed", "tasks_unsupported"]
tbl[show].round(3)

In [ ]:
Path("paper/tables").mkdir(parents=True, exist_ok=True); Path("paper/figures").mkdir(parents=True, exist_ok=True)
open("paper/tables/main.tex", "w").write(ag.to_latex(tbl, ["mase_rel", "wql_rel", "skill_%", "avg_rank", "win_rate_%"], caption="Main results.", label="tab:main"))
fig = plots.bar_relative(tbl, title="All methods vs seasonal naive"); fig.savefig("paper/figures/main_bar.pdf", bbox_inches="tight"); plt.show()

**How to read:** `< 1` beats seasonal naive. Methods whose CI includes 1 are not distinguishable from it. `tasks_unsupported` > 0 means the method
could not run on those tasks (e.g. horizon cap) and was imputed with seasonal naive — report coverage next to every score.

## 2 · Statistical significance

In [ ]:
w = ag.wide(df, "mase")
cd = ag.friedman_nemenyi(w[[c for c in w.columns if c != "seasonal_naive"]])
print({k: v for k, v in cd.items() if k != "avg_ranks"}); fig = plots.critical_difference(w); fig.savefig("paper/figures/cd.pdf", bbox_inches="tight"); plt.show()

Pairwise win-rate (row beats column, fraction of tasks):

In [ ]:
wr = ag.pairwise_winrate(w); import seaborn as sns
plt.figure(figsize=(7, 5.5)); sns.heatmap(wr.astype(float), annot=True, fmt=".2f", cmap="RdBu", center=.5, cbar=False); plt.savefig("paper/figures/winrate.pdf", bbox_inches="tight"); plt.show()

## 3 · RQ1 — which family wins, and where?

In [ ]:
fam = df[df.status == "ok"].merge(tbl[["family"]].reset_index().rename(columns={"family": "fam"}), on="model")
for by in ["term", "freq", "domain"]:
    if df[by].notna().any():
        g = ag.group_table(df, by); display(g.round(3)); plots.group_heatmap(g, title=f"rel. MASE by {by}").savefig(f"paper/figures/by_{by}.pdf", bbox_inches="tight")

## 4 · RQ2 — do LLM weights help? (Tan et al. replication on new data)
Compare `gpt4ts` / `time_llm` against `tan_*` and `gpt2_scratch`. If `tan_wo_llm` ≈ `gpt4ts`, pretrained language weights add nothing on this benchmark.

In [ ]:
rq2 = [m for m in ["gpt4ts", "time_llm", "tan_wo_llm", "tan_llm2attn", "tan_llm2trsf", "gpt2_scratch", "patchtst"] if m in w.columns]
if len(rq2) >= 2:
    print(tbl.loc[rq2, ["mase_rel", "avg_rank"]].round(3))
    for a in rq2[1:]: print(f"gpt4ts vs {a}: Wilcoxon p = {ag.paired_wilcoxon(w, rq2[0], a):.3f}") if rq2[0] in w else None
else: print("run the LLM-reuse and ablation models first")

## 5 · RQ3 — accuracy vs cost

In [ ]:
plots.accuracy_vs_cost(tbl, title="Accuracy vs inference time").savefig("paper/figures/pareto.pdf", bbox_inches="tight"); plt.show()
tbl[["fit_s_median", "predict_s_median"]].round(2)

## 6 · Contamination robustness
Re-compute the main table excluding (model, task) pairs flagged in `configs/contamination.csv`/task meta. Rankings that survive are the ones to trust.

In [ ]:
clean = df[~df.contaminated.fillna(False)]
print("flagged rows:", int(df.contaminated.fillna(False).sum()), "of", len(df))
ag.summary_table(clean)[["mase_rel", "avg_rank"]].round(3).join(tbl[["mase_rel", "avg_rank"]], rsuffix="_all")